# Divya — RLCD fine-tune of System-1 (Laya) on Indian corporate filings

Runs on Kaggle's free **GPU T4 x2**, which is the exact hardware Laya's own reference
recipe (`laya_finetune_typed_decisions_2xT4_kaggle.ipynb`) specifies. Everything expensive
happens here; the local machine only writes code and scrapes small JSON.

**What this decides.** `docs/loop/STOP_RULE.md` pre-commits, before any training:

> If a fine-tuned Laya does not reach **≥ 0.65 macro-F1 on the 120 held-out NSE
> announcements**, the typed-decision approach is refuted for this domain.

The 120 are excluded from the corpus **by id** and are evaluated exactly once, at the end.

**Baseline to beat:** zero-shot macro-F1 **0.147** / accuracy **0.513** on the dev split,
and **0.433** accuracy on the held-out 120 through the Router.

The four failed question-set experiments (D-024 → D-026) established that the limit is the
model, not the prompt. This is the test of that conclusion.

In [ ]:
import os, sys, json, time, glob, shutil, zipfile
!pip install -q laya transformers safetensors huggingface_hub bitsandbytes 2>&1 | tail -1
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  gpu{i}: {torch.cuda.get_device_name(i)} '
              f'{torch.cuda.get_device_properties(i).total_memory/2**30:.1f} GB')
else:
    print('  NO GPU -- this kernel must run with GPU T4 x2 enabled')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)


# Resolve the dataset directory by CONTENT, not by path pattern. Kaggle creates several
# directories under /kaggle/input -- including a bare 'datasets' owner folder -- and a glob
# for the dataset name matched the wrong one. Asking 'which directory actually holds the
# corpus' cannot be wrong.
def find_dir(filename, roots=None):
    roots = roots or [d for d in glob.glob('/kaggle/input/*') if os.path.isdir(d)]
    roots += [os.getcwd(), '/kaggle/working']
    for r in roots:
        if os.path.isfile(os.path.join(r, filename)):
            return r
    for r in roots:
        for dirpath, _dirs, files in os.walk(r):
            if filename in files:
                return dirpath
    raise SystemExit(f'could not locate {filename} under {roots}')

DATASET_DIR = find_dir('finetune_corpus.jsonl')
print('dataset dir:', DATASET_DIR)
print('contents:', sorted(os.listdir(DATASET_DIR))[:10])

# The trainer lives in the dataset, not in the working directory.
if os.path.isfile(os.path.join(DATASET_DIR, 'finetune_laya.py')):
    shutil.copy(os.path.join(DATASET_DIR, 'finetune_laya.py'), '/kaggle/working/')
print('trainer present:', os.path.isfile('/kaggle/working/finetune_laya.py'))


In [ ]:
base = (f'python /kaggle/working/finetune_laya.py --eval-only --limit-dev 200 '
        f'--device {DEVICE} '
        f'--corpus {DATASET_DIR}/finetune_corpus.jsonl '
        f'--heldout {DATASET_DIR}/nse_v5_eval_set.jsonl')
print(base)
!{base}


## Cell 3 — zero-shot baseline on dev, for the record

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working')
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

base = !python finetune_laya.py --eval-only --limit-dev 200 --device cuda \
      --corpus {DATASET_DIR}/finetune_corpus.jsonl \
      --heldout {DATASET_DIR}/nse_v5_eval_set.jsonl

## Cell 4 — the fine-tune

Single process with gradient accumulation, rather than the reference's DDP. T4 x2 is two
separate cards and DDP buys throughput, not quality; one card at micro-batch 8 reproduces
the reference's effective batch without the added failure surface. Effective batch here is
**64**, matching the reference.

In [ ]:
train_cmd = (
    f'python finetune_laya.py --epochs 4 --micro-batch 8 --accum 8 '
    f'--group-size 4 --device {DEVICE} '
    f'--corpus {DATASET_DIR}/finetune_corpus.jsonl '
    f'--heldout {DATASET_DIR}/nse_v5_eval_set.jsonl '
    f'--out /kaggle/working/finetuned'
)
print(train_cmd)
t0 = time.time()
!{train_cmd}
print(f'\nTRAINING WALL TIME: {time.time()-t0:.0f}s')

## Cell 5 — evaluate ONCE on the 120 held-out announcements

This is the number that decides the project. It is computed once. If it misses the bar, the
stop rule fires and we do not tune against this set again.

In [ ]:
import json, torch
from safetensors.torch import load_file
from transformers import AutoTokenizer
from laya.common import build_model, QTYPES, build_sequence, render_options

OUT = '/kaggle/working/finetuned'
cfg = json.load(open(f'{OUT}/rl_agent_config.json'))
tok = AutoTokenizer.from_pretrained(f'{OUT}/tokenizer')
model = build_model(cfg, encoder_dir=f'{OUT}/encoder')
model.load_state_dict(load_file(f'{OUT}/model.safetensors'), strict=True)
model.to('cuda').eval()

rows = [json.loads(l) for l in open(f'{DATASET_DIR}/nse_v5_eval_set.jsonl') if l.strip()]
options = sorted({r['labels']['event_type'] for r in rows})
crit = {o: '' for o in options}
q = {'t': 'choice', 'ins': 'Classify the corporate event this filing discloses.', 'crit': crit}
k = len(render_options(q))

from collections import Counter
tot, ok = Counter(), Counter()
with torch.no_grad():
    for r in rows:
        seq, markers = build_sequence(tok, r['content'], q, cfg['max_len'], cfg['head_max_len'])
        if len(markers) != k:
            continue
        ids = torch.tensor([seq], device='cuda'); att = torch.ones_like(ids)
        mpos = torch.tensor([markers], device='cuda')
        mmask = torch.ones_like(mpos, dtype=torch.bool)
        qt = torch.tensor([QTYPES['choice']], device='cuda')
        logits, _ = model(ids, att, mpos, mmask, qt)
        pred = options[int(logits.float().reshape(-1).argmax())]
        gold = r['labels']['event_type']
        tot[gold] += 1
        ok[gold] += (pred == gold)

f1s = []
for lb in options:
    tp = ok[lb]; fn = tot[lb]-tp; fp = (sum(tot.values())-tot[lb])-tp
    p = tp/(tp+fp) if tp+fp else 0.0; rc = tp/(tp+fn) if tp+fn else 0.0
    f1s.append(2*p*rc/(p+rc) if p+rc else 0.0)
n = max(sum(tot.values()), 1)
macro_f1 = sum(f1s)/len(f1s)
acc = sum(ok.values())/n

print(f'HELD-OUT n={n}  accuracy={acc:.4f}  macro_f1={macro_f1:.4f}')
for lb in options:
    print(f'  {lb:<20} n={tot[lb]:<3} acc={ok[lb]/tot[lb]:.3f}')
BAR = 0.65
print(f'\nSTOP RULE (>= {BAR} macro-F1): ' + ('PASSED' if macro_f1>=BAR else 'FIRED'))
json.dump({'n': n, 'accuracy': acc, 'macro_f1': macro_f1, 'bar': BAR,
           'passed': macro_f1 >= BAR,
           'per_class': {lb: round(ok[lb]/tot[lb],4) for lb in options}},
          open('/kaggle/working/HELD_OUT_RESULT.json','w'), indent=2)

## Cell 6 — export the model

In [ ]:
!cd /kaggle/working/finetuned && zip -qr /kaggle/working/laya-indian-filings.zip . && ls -lh /kaggle/working/laya-indian-filings.zip
print('output dir: /kaggle/working')